# Spam Email Classifier
TF-IDF + Naive Bayes / Logistic Regression / Linear SVM on the Enron spam/ham dataset.

In [ ]:
%pip install pandas scikit-learn nltk matplotlib

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

STEP 1 : LOAD THE DATA


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/nikhil1102/Spam-Classifier/main/spam_ham_dataset.csv"
df = pd.read_csv(url)
df = df[['label', 'text']]   # drop the unnamed index column, keep what we need

print(df.shape)
df.head()

(5171, 2)


,label,text
0,ham,Subject: enron methanol ; meter # : 988291\r\n...
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see..."
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar..."
3,spam,"Subject: photoshop , windows , office . cheap ..."
4,ham,Subject: re : indian springs\r\nthis deal is t...


STEP 2 : EXPLORE THE DATA (EDA)

In [ ]:
print(df['label'].value_counts())
print(df.isnull().sum())

df['length'] = df['text'].apply(len)
print(df.groupby('label')['length'].mean())

label
ham     3672
spam    1499
Name: count, dtype: int64
label    0
text     0
dtype: int64
label
ham      977.008170
spam    1223.256171
Name: length, dtype: float64


STEP 3 : CLEAN THE DATA

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\S+@\S+', ' ', text)        # remove email addresses
    text = re.sub(r'http\S+|www\S+', ' ', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)         # remove punctuation & numbers
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    words = [stemmer.stem(w) for w in words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)
print(df[['text', 'clean_text']].iloc[0])

text          Subject: enron methanol ; meter # : 988291\r\n...
clean_text    subject enron methanol meter follow note gave ...
Name: 0, dtype: object


STEP 4 : TRAIN/TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(4136,) (1035,)


STEP 5 : CONVERT TEXT INTO NUMBER

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(X_train_vec.shape)

(4136, 3000)


STEP 6 : TRAIN CLASSIFIER

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC()
}

trained = {}
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    trained[name] = model

STEP 7 : EVALUATE

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

for name, model in trained.items():
    y_pred = model.predict(X_test_vec)
    print(f"--- {name} ---")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1-score :", f1_score(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print()

--- Naive Bayes ---
Accuracy : 0.9487922705314009
Precision: 0.8871473354231975
Recall   : 0.9433333333333334
F1-score : 0.9143780290791599
[[699  36]
 [ 17 283]]

--- Logistic Regression ---
Accuracy : 0.9768115942028985
Precision: 0.9539473684210527
Recall   : 0.9666666666666667
F1-score : 0.9602649006622517
[[721  14]
 [ 10 290]]

--- Linear SVM ---
Accuracy : 0.9806763285024155
Precision: 0.9605263157894737
Recall   : 0.9733333333333334
F1-score : 0.9668874172185431
[[723  12]
 [  8 292]]



STEP 8 : TEST YOUR OWN EMAIL TEXT

In [ ]:
def predict_email(text, model_name="Linear SVM"):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = trained[model_name].predict(vec)[0]
    return "SPAM" if pred == 1 else "HAM"

print(predict_email("Subject: You won a free prize, click here to claim now!!!"))
print(predict_email("Subject: Meeting rescheduled to 3pm tomorrow, see attached agenda."))

SPAM
HAM


STEP 9 : INTERACTIVE PREDICTION

In [ ]:
def predict_email(text, model_name="Linear SVM"):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = trained[model_name].predict(vec)[0]
    return "SPAM" if pred == 1 else "HAM"

# --- Interactive loop ---
print("Email Spam Classifier — type 'exit' to quit")
while True:
    user_input = input("\nEnter an email/message: ")
    if user_input.lower() == "exit":
        break
    result = predict_email(user_input)
    print(f"Prediction: {result}")

Email Spam Classifier — type 'exit' to quit

Enter an email/message: free tickets
Prediction: HAM

Enter an email/message: sell sell
Prediction: SPAM

Enter an email/message: exit
